In [ ]:
'''
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import shutil

target = Path('./data/2_frame_files')
subdirs = sorted([d for d in target.iterdir() if d.is_dir()])
print(subdirs[:5])

def rm_rf(path):
    subprocess.run(['rm', '-rf', str(path)])

for subdir in tqdm(subdirs, desc="삭제 중"):
    # 하위폴더 내부의 폴더들을 병렬로 rm -rf
    inner_dirs = [d for d in subdir.iterdir() if d.is_dir()]
    if inner_dirs:
        with ThreadPoolExecutor(max_workers=8196) as pool:
            list(pool.map(rm_rf, inner_dirs))
    shutil.rmtree(subdir)

# 남은 최상위 파일 및 폴더 정리
subprocess.run(['rm', '-rf', str(target)])
'''

In [ ]:
import requests
try:
    print(requests.get("http://localhost:8000/health", timeout=5).text)
except:
    print("❌ 서버 죽음 → 셀 2 다시 실행")

In [ ]:
# %% parquet 확인
import pandas as pd
import json
import glob

parquets = glob.glob("./data/4_telop_results/*.parquet")
print(f"총 {len(parquets)}개 채널 파일\n")

n = 10
df = pd.read_parquet(parquets[n])
print(f"📁 {parquets[n]}")
print(f"   shape: {df.shape}\n")
print(df.head(10).to_string())

In [ ]:
df.head()

In [ ]:
len(df)

In [ ]:
# %% parquet 검증
import os
import json
import glob
import pandas as pd

OUTPUT_DIR = "./data/4_telop_results"
parquets = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.parquet")))
print(f"총 {len(parquets)}개 채널 파일\n")

total_rows = 0
issues = []

for pq in parquets:
    name = os.path.basename(pq).replace(".parquet", "")
    try:
        df = pd.read_parquet(pq)
        n = len(df)
        total_rows += n

        # 필수 컬럼 확인
        expected = {"video_name", "frame_num", "ocr_xywha", "ocr_texts", "is_telop"}
        missing = expected - set(df.columns)
        if missing:
            issues.append(f"  {name}: 컬럼 누락 {missing}")
            continue

        # is_telop 파싱 + None 체크
        n_none = 0
        n_parse_err = 0
        for _, row in df.iterrows():
            try:
                vals = json.loads(row["is_telop"])
                if any(v is None for v in vals):
                    n_none += 1
            except:
                n_parse_err += 1

        status = f"✅ {name}: {n}개 영상"
        if n_none:
            status += f" | ⚠️ is_telop=None {n_none}건(에러)"
        if n_parse_err:
            status += f" | ❌ 파싱실패 {n_parse_err}건"
        print(status)

    except Exception as e:
        issues.append(f"  {name}: 파일 읽기 실패 — {e}")

print(f"\n총 {total_rows}개 영상 저장됨")
if issues:
    print(f"\n❌ 문제 파일 ({len(issues)}건):")
    for i in issues:
        print(i)
else:
    print("문제 없음 👍")

In [ ]:
# %% parquet 확인
import pandas as pd
import json
import glob

parquets = glob.glob("./data/4_is_telop_results/*.parquet")
print(f"총 {len(parquets)}개 채널 파일\n")

n = 0
df = pd.read_parquet(parquets[n])
print(f"📁 {parquets[n]}")
print(f"   shape: {df.shape}\n")
print(df.head(50).to_string())

In [1]:
# %% 토큰 분포 확인 (32코어 병렬)
import json
import numpy as np
from transformers import AutoTokenizer
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm

JSONL_PATH = "./data/5_qwen_train_dataset.jsonl"
MODEL_ID = "Qwen/Qwen3.5-0.8B"

# 먼저 전체 라인 로드
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()
print(f"📄 {len(lines):,}개 샘플 로드")


def count_tokens(line):
    tokenizer = getattr(count_tokens, "_tok", None)
    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        count_tokens._tok = tokenizer
    sample = json.loads(line)
    text = tokenizer.apply_chat_template(sample["messages"], tokenize=False)
    return len(tokenizer.encode(text))


with ProcessPoolExecutor(max_workers=32, initializer=lambda: None) as executor:
    token_counts = list(tqdm(executor.map(count_tokens, lines, chunksize=64), total=len(lines), desc="토큰 수 계산"))

arr = np.array(token_counts)
print(f"\n샘플 수: {len(arr):,}")
print(f"min:  {arr.min():,}")
print(f"p25:  {int(np.percentile(arr, 25)):,}")
print(f"p50:  {int(np.percentile(arr, 50)):,}")
print(f"p75:  {int(np.percentile(arr, 75)):,}")
print(f"p90:  {int(np.percentile(arr, 90)):,}")
print(f"p95:  {int(np.percentile(arr, 95)):,}")
print(f"p99:  {int(np.percentile(arr, 99)):,}")
print(f"max:  {arr.max():,}")
print(f"mean: {arr.mean():,.0f}")

📄 66,400개 샘플 로드


토큰 수 계산:   0%|          | 0/66400 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (288886 > 262144). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (386957 > 262144). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (264259 > 262144). Running this sequence through the model will result in indexing errors



샘플 수: 66,400
min:  346
p25:  6,457
p50:  13,917
p75:  25,653
p90:  38,278
p95:  48,414
p99:  81,319
max:  386,957
mean: 18,580
